# Sleep Disorder Detection
### 3D CNN Pipeline Execution Hub

This notebook serves as the unified entry point for running the Sleep Disorder Detection pipeline. It handles:
1. **Data Building**: Processing raw EEG (EDF) files into 4D spatiotemporal tensors.
2. **Model Training**: Training the 3D CNN on the processed dataset.
3. **Evaluation**: Visualizing results and performance metrics.

In [1]:
import os
import sys
from pathlib import Path

# Add project root to path for imports
project_root = str(Path(os.getcwd()).parent.parent)
if project_root not in sys.path:
    sys.path.append(project_root)

print(f"Project Root: {project_root}")

Project Root: /Users/pawankumar/SleepDisorderDetection


## 1. Dataset Construction

In this step, we convert raw EEG recordings and their corresponding sleep stage annotations into spatiotemporal tensors. Each tensor captures the scalp distribution of **Delta**, **Alpha**, and **Beta** band power over a 30-second epoch.

In [2]:
from src.data.builder import build_dataset
from config.constants import DATASET_ROOT

# --- CONFIGURATION ---
DATA_DIR = "/Users/pawankumar/SleepDisorderDetection/raw_data/sample" # CHANGE THIS to your raw data path containing .edf and .edf.st
SAMPLE_NORMS = 30              # Number of patients to sample for global normalization

if not os.path.exists(DATA_DIR):
    print(f"⚠️  WARNING: Data directory not found: {DATA_DIR}")
    print("Please update DATA_DIR before running the cell below.")

In [3]:
# RUN DATASET BUILD
try:
    build_dataset(
        data_dir=DATA_DIR, 
        output_dir=os.path.join(project_root, DATASET_ROOT), 
        sample_norms=SAMPLE_NORMS
    )
    print("\n✅ Dataset build completed successfully!")
except Exception as e:
    print(f"\n❌ Error during dataset build: {e}")

[builder] Found 5 patients.
[builder] Loading existing norms from /Users/pawankumar/SleepDisorderDetection/dataset/metadata/global_norms.npy

[1/5] Progress...

[builder] Processing Patient: brux1 (Disorder: 7)
  [signal] Loaded 7 channels. Duration: 14341.0s
  [parser] Parsed 354 stage labels and 60 CAP events.

[2/5] Progress...

[builder] Processing Patient: ins1 (Disorder: 4)
  [signal] Loaded 4 channels. Duration: 48000.0s
  [parser] Parsed 900 stage labels and 315 CAP events.

[3/5] Progress...

[builder] Processing Patient: n1 (Disorder: 0)
  [signal] Loaded 9 channels. Duration: 1023.0s
  [parser] Parsed 1132 stage labels and 530 CAP events.

[4/5] Progress...

[builder] Processing Patient: narco1 (Disorder: 5)
  [signal] Loaded 4 channels. Duration: 33840.0s
  [parser] Parsed 1093 stage labels and 222 CAP events.

[5/5] Progress...

[builder] Processing Patient: nfle1 (Disorder: 1)
  [signal] Loaded 7 channels. Duration: 575.0s
  [parser] Parsed 947 stage labels and 595 CAP ev

/Users/pawankumar/SleepDisorderDetection/src/data/builder.py:192: RuntimeWarning: divide by zero encountered in divide
  weights = total / (n_classes * counts.astype(float))


## 2. Model Training

Once the dataset is built, we train a 3D CNN model to classify epochs into sleep disorder categories. The training process includes class balancing (to handle imbalanced data) and early stopping.

In [4]:
from scripts.train_model import main as run_training

# Change directory to project root to ensure relative paths in constants.py work
os.chdir(project_root)

print("Starting model training...")
try:
    run_training()
    print("\n✅ Training sequence finished!")
except Exception as e:
    print(f"\n❌ Training failed: {e}")

2026-04-08 15:57:10 | training | [INFO] | ============================================================
2026-04-08 15:57:10 | training | [INFO] | SLEEP DISORDER DETECTION — MODEL TRAINING
2026-04-08 15:57:10 | training | [INFO] | ============================================================
2026-04-08 15:57:10 | training | [INFO] | Device: mps
2026-04-08 15:57:10 | training | [INFO] | [step 1] Loading Dataset...
2026-04-08 15:57:10 | training | [INFO] | [step 2] Splitting Data (Stratified)...
2026-04-08 15:57:10 | training | [INFO] | [info] Using balanced class weights.
2026-04-08 15:57:10 | training | [INFO] | [step 3] Initializing Model...
2026-04-08 15:57:10 | training | [INFO] |   Architecture: 3D CNN
2026-04-08 15:57:10 | training | [INFO] |   Parameters  : 1,206,180


Starting model training...
  [dataset] Initialized with 1824 epochs and 4 classes.
  [split] ⚠️ Stratification failed (likely rare classes in small sample set). Falling back to random split.

  [split] Train: 1276, Val: 274, Test: 274


2026-04-08 15:57:11 | training | [INFO] | [step 4] Starting Training...
2026-04-08 16:03:25 | training | [INFO] |   Epoch   1/100 | Train Loss: 1.3249 Acc: 0.574 | Val Loss: 1.2926 Acc: 0.376


: 

## 3. Performance & Visualization

The training script automatically saves plots to `training_output/`. You can view them here.

In [ ]:
from IPython.display import Image, display
import glob

output_dir = "training_output"
plots = glob.glob(f"{output_dir}/*.png")

if plots:
    for plot_path in plots:
        print(f"--- {os.path.basename(plot_path)} ---")
        display(Image(filename=plot_path))
else:
    print("No plots found. Ensure training has completed.")